# 02 – Cleaning and feature engineering
Goal: turn the raw London listings data into a clean table of features for predicting nightly price, and save it for modelling.

In [1]:
import json
import numpy as np
import pandas as pd

df = pd.read_csv("../data/raw/listings.csv.gz")
print(df.shape)
print(df["neighbourhood_cleansed"].nunique(), "neighbourhoods")

(92638, 90)
33 neighbourhoods


In [2]:
df["price_num"] = df["price"].str.replace(r"[£$,]", "", regex=True).astype(float)
df = df.dropna(subset=["price_num"])
df = df[(df["price_num"] >= 20) & (df["price_num"] <= 1000)].copy()
print(df.shape)


(60852, 91)


In [3]:
df["log_price"] = np.log(df["price_num"])
df[["price_num", "log_price"]].describe().round(2)


,price_num,log_price
count,60852.00,60852.00
mean,223.47,5.14
std,171.02,0.74
min,20.00,3.00
25%,99.00,4.60
50%,177.00,5.18
75%,290.41,5.67
max,1000.00,6.91


In [4]:
print("Bathrooms missing:", df["bathrooms"].isna().sum())
print("Bathrooms text missing:", df["bathrooms_text"].isna().sum())
df["bathrooms_text"].value_counts().head(15)

Bathrooms missing: 6421
Bathrooms text missing: 107


bathrooms_text
1 bath               27625
2 baths               8508
1 shared bath         7008
1 private bath        6116
1.5 baths             3226
2.5 baths             1700
1.5 shared baths      1604
3 baths               1158
2 shared baths        1053
3.5 baths              414
Shared half-bath       389
0 baths                319
0 shared baths         265
2.5 shared baths       242
Private half-bath      217
Name: count, dtype: int64

In [5]:
def parse_baths(text):
    if pd.isna(text):
        return np.nan
    text = text.lower()
    if "half-bath" in text:
        return 0.5
    try:
        return float(text.split()[0])
    except ValueError:
        return np.nan

df["baths_from_text"] = df["bathrooms_text"].apply(parse_baths)
before = df["bathrooms"].isna().sum()
df["bathrooms"] = df["bathrooms"].fillna(df["baths_from_text"])
print(f"Missing bathrooms: {before} before, {df['bathrooms'].isna().sum()} after")

df["bath_shared"] = (
    df["bathrooms_text"].str.contains("shared", case=False, na=False).astype(int)
)
df["bath_shared"].value_counts()

Missing bathrooms: 6421 before, 99 after


bath_shared
0    50012
1    10840
Name: count, dtype: int64

In [6]:
centre_lat, centre_lon = 51.5080, -0.1281  # Trafalgar Square

lat1, lon1 = np.radians(df["latitude"]), np.radians(df["longitude"])
lat2, lon2 = np.radians(centre_lat), np.radians(centre_lon)
a = (np.sin((lat2 - lat1) / 2) ** 2
     + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2)
df["dist_centre_km"] = 6371 * 2 * np.arcsin(np.sqrt(a))

df["dist_centre_km"].describe().round(1)

count    60852.0
mean         7.7
std          5.1
min          0.1
25%          3.9
50%          6.3
75%         10.6
max         28.8
Name: dist_centre_km, dtype: float64

In [7]:
df["has_reviews"] = (df["number_of_reviews"] > 0).astype(int)

def count_amenities(text):
    if pd.isna(text):
        return 0
    return len(json.loads(text))

df["amenities_count"] = df["amenities"].apply(count_amenities)

df["is_superhost"] = (df["host_is_superhost"] == "t").astype(int)
df["instant_bookable"] = (df["instant_bookable"] == "t").astype(int)

df[["has_reviews", "amenities_count", "is_superhost", "instant_bookable"]].describe().round(2)

,has_reviews,amenities_count,is_superhost,instant_bookable
count,60852.00,60852.00,60852.00,60852.0
mean,0.79,31.59,0.24,0.0
std,0.41,15.07,0.43,0.0
min,0.00,0.00,0.00,0.0
25%,1.00,20.00,0.00,0.0
50%,1.00,33.00,0.00,0.0
75%,1.00,42.00,0.00,0.0
max,1.00,103.00,1.00,0.0


In [8]:
feature_cols = [
    "neighbourhood_cleansed", "room_type", "accommodates", "bedrooms",
    "beds", "bathrooms", "bath_shared", "latitude", "longitude",
    "dist_centre_km", "minimum_nights", "number_of_reviews",
    "has_reviews", "review_scores_rating", "amenities_count",
    "is_superhost", "instant_bookable",
]
target_cols = ["price_num", "log_price"]

clean = df[feature_cols + target_cols].copy()
print(clean.shape)
clean.isna().mean().round(3).sort_values(ascending=False).head(8)

(60852, 19)


bedrooms                  0.224
review_scores_rating      0.212
beds                      0.068
bathrooms                 0.002
neighbourhood_cleansed    0.000
number_of_reviews         0.000
price_num                 0.000
instant_bookable          0.000
dtype: float64

In [9]:
clean.select_dtypes("number").corr()["log_price"].drop(["log_price", "price_num"]).sort_values(ascending=False).round(2)

accommodates            0.61
beds                    0.47
bedrooms                0.44
amenities_count         0.33
bathrooms               0.31
is_superhost            0.07
review_scores_rating    0.05
latitude                0.01
has_reviews            -0.02
minimum_nights         -0.09
longitude              -0.09
number_of_reviews      -0.11
dist_centre_km         -0.36
bath_shared            -0.52
instant_bookable         NaN
Name: log_price, dtype: float64

In [10]:
clean.to_csv("../data/processed/listings_clean.csv", index=False)
print("Saved", clean.shape)

Saved (60852, 19)


## Cleaning decisions
- Kept listings priced £20–£1,000, as in the exploration notebook. The model predicts log(price).
- Filled missing bathroom counts using the bathrooms text column, and added a feature for whether the bathroom is shared.
- Added distance to central London (Trafalgar Square) using the haversine formula.
- Added features for whether a listing has reviews, its number of amenities, superhost status and instant booking.
- Excluded columns calculated from price (such as estimated revenue) to avoid data leakage, and excluded personal data such as host names.
- Remaining missing values (e.g. bedrooms, beds, review scores) will be filled during modelling, using only the training data.